# From whole-slide images to tile embeddings with UNI2-h

**Goal.** Turn each whole-slide image (WSI) of the BRACS dataset into a set of fixed-size tile embeddings produced by the pathology foundation model UNI2-h, so that they can be used later for classification.

**Pipeline.**
1. **Setup**: library versions, compute device and imports.
2. **Data**: parameters, tissue filter, tile index, visual check and data loading.
3. **Feature extraction**: load UNI2-h and save one feature file per slide.
4. **Inspection**: check one feature file, then the embeddings of all slides.

**Inputs / outputs.** Input: `.svs` slides in `DATA_DIR`. Output: one `.pt` file per slide in `FEATURES_DIR/<model>/`, holding the CLS vector of every tile, the tile coordinates and the tiling parameters (and optionally the patch tokens).

**Reproducibility.** Library versions are printed in section 1.1. All tiling parameters are collected in section 2.1 and stored in every output file.

## 1. Setup

### 1.1 Environment and library versions
Prints the Python and library versions used in this run, so results can be traced back to the software that produced them.

In [1]:
# STEP 1: Import general-purpose third-party libraries (tensors, plotting, data handling)
import torch
import torchvision
import pandas as pd
import numpy as np
import sys

# STEP 2: Print library versions for reproducibility
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)

System Version: 3.14.7 (main, Aug 10 2026, 07:46:56) [GCC 16.1.1 20260728]
PyTorch version: 2.15.0+cu132
Torchvision version: 0.29.2+cu132
Numpy version: 2.5.4
Pandas version: 3.1.1


### 1.2 Compute device
Selects the fastest available device (CUDA, then Apple MPS, then CPU) and enables cuDNN autotuning on GPU. This is safe here because every input tile has the same size.

In [2]:
# STEP 1: Select CUDA, then Apple MPS, then CPU. Pass e.g. get_device("cpu") to force one.
from utils.device import get_device

device = get_device()

# STEP 2: Enable cuDNN autotuning when on GPU (safe here since input tile size is fixed)
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print(f"Using {device.type.upper()}")

# Expose the device to the notebook
device

Using CUDA: NVIDIA RTX PRO 4500 Blackwell


device(type='cuda')

### 1.3 Imports
Imports the project's own building blocks from `utils/`: dataset and tile filter, transforms, data loader, model loading, feature extraction and the inspection plots.

In [ ]:
# STEP 1: Standard-library and plotting imports
from functools import partial
from pathlib import Path
from time import time
import matplotlib.pyplot as plt

# STEP 2: Import this project's own pipeline building blocks (dataset, filters,
# transforms, feature extraction, model loading, inspection plots)
from utils.dataset import WSITileDataset
from utils.clean_empty import background_filter
from utils.transformer_basic import build_basic_transform
from utils.dataloader import create_dataloader
from utils.transformer_per_model import build_uni2h_transform
from utils.load_model import load_model, MODEL_NAME
from utils.generate_features import generate_features
from utils.plot_wsi import plot_wsi
from utils.inspect_features import inspect_feature_file
from utils.cls_overview import plot_cls_overview

## 2. Data

### 2.1 Experiment parameters
All tiling settings in one place.
- `TILE_SIZE = 224` is the input size of UNI2-h. With `STRIDE == TILE_SIZE` the tiles do not overlap.
- `PYRAMID_LEVEL` selects the resolution: level 1 is full resolution, higher levels are more downsampled. The number of tiles grows roughly 16-fold for each level step towards level 1.
- `BATCH_SIZE` is the number of tiles per model call.

In [4]:
# Major variables
BATCH_SIZE = 32          # STEP 1: Tiles per batch fed through the model
# TODO: check input to the models if they come from here
TILE_SIZE = 224          # STEP 2: Width/height (in pixels) of each square tile
STRIDE = 224             # STEP 3: Distance between neighboring tile origins; STRIDE == TILE_SIZE -> no overlap
DATA_DIR='/mnt/Data/bracs/data'  # STEP 4: Directory of source WSI TIFFs
PYRAMID_LEVEL = 2        # STEP 5: Which pyramid resolution level to tile from (higher = more downsampled)
# TODO: Think whether it makes sense to save these parameters in/on the output file
# TODO: make "brackets" para train/val/test with same pacients
# TODO: think whether it makes sense to use the augmentation flags

### 2.2 Tile filter, transform and dataset index
- **Tile filter:** a tile is kept only if at least 50% of its pixels are neither near-white nor near-black, which removes background and empty glass.
- **Basic transform:** model-independent preprocessing; augmentations are switched off because this run only extracts features.
- **Dataset:** building it indexes every tile of every slide (in parallel) and applies the filter. Slides without the requested level, and tiles that cannot be read, are left out and reported as warnings.

In [5]:
# STEP 1: Define the tissue/background filter — a tile is kept only if at least
# min_tissue_fraction of its pixels are neither near-white nor near-black
tile_filter = partial(
    background_filter,
    min_tissue_fraction=0.5,
    white_threshold=220,
    black_threshold=15,
)
print("Tile filter created")

# STEP 2: Define the shared, model-agnostic transform applied to every tile as it's
# loaded (to-tensor/dtype scaling here; augmentations disabled for this inspection run)
basic_transform = build_basic_transform(
    apply_color_jitter=False,
    apply_horizontal_flip=False,
    apply_vertical_flip=False,
    apply_rotation=False,
)
print("Basic transform created")

# STEP 3: Index every tile across every slide in DATA_DIR at the target pyramid level,
# applying tile_filter during indexing so only tissue-containing tiles are kepts
dataset = WSITileDataset(
    data_dir=DATA_DIR,
    level=PYRAMID_LEVEL,
    tile_size=TILE_SIZE,
    stride=STRIDE,
    basic_transform=basic_transform,
    tile_filter=tile_filter,
)
print("Dataset created")


Tile filter created
Basic transform created
Dataset created


/home/guisazevedo/Projects/01_accamargo/phd_streamembed/utils/dataset.py:276: UserWarning: Left out unreadable tiles in 1 slide(s): BRACS_1791.svs (4 tiles)
  warnings.warn(


### 2.3 Dataset size
Number of tiles that passed the filter and number of slides skipped because they lack the requested pyramid level. Check this against the expected cohort size before the long run.

In [6]:
print(len(dataset), "tiles;", len(dataset.skipped_slides), "slides skipped")

1457195 tiles; 0 slides skipped


### 2.4 Visual check of the tiling
For each chosen slide, `plot_wsi` shows (top) the slide at the tiled level, (middle) the tile grid with the tiles that passed the filter outlined in blue and the first 12 outlined in orange, and (bottom) those 12 tile crops as the dataset returns them.

Check that tissue is kept, glass is dropped and the crops look like what the model should see. Large pyramid levels take several seconds per slide to draw.

In [ ]:
# STEP 1: Choose the slides to draw (position among the slides that have tiles, or a file name)
SLIDES_TO_PLOT = [0, 1, 2]

# STEP 2: Draw each slide with its tile grid and its first tile crops
for slide in SLIDES_TO_PLOT:
    _ = plot_wsi(dataset, slide=slide, n_tiles=12)

### 2.5 Data loader
Batches the tiles for inference. `shuffle=False` is required: features are written slide by slide, so the tiles of one slide must arrive together and in order. Several worker processes read tiles while the model runs.

In [8]:
# STEP 1: Build the DataLoader that batches tiles for model inference below
dataloader = create_dataloader(
    dataset = dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    device=device,
)

## 3. Feature extraction

### 3.1 Load the model
Builds the UNI2-h architecture, loads the local checkpoint from `original_vit_model/` and sets it to evaluation mode. The matching model-specific transform (resize and ImageNet normalization) is created alongside.

In [9]:
# STEP 1: Load the UNI2-h model (weights in original_vit_model/) and its transform
transform = build_uni2h_transform()
model = load_model(device=device)

Loading model: UNI2-h
Model loaded: UNI2-h
Model moved to device: cuda


### 3.2 Extract and save features
Runs every tile through UNI2-h and writes one `.pt` file per slide to `FEATURES_DIR/<model>/<slide>.pt`, containing `cls_features`, `coordinates`, `model_encoder`, `tile_params` and, only if `save_patch_features=True`, `patch_features`.

CLS-only files are about 250 times smaller than files with patch tokens. This is the long-running step; the printed timing allows extrapolating to other pyramid levels.

In [10]:
# STEP 1: Directory where every model's features will be written, one subfolder per model
from pathlib import Path

FEATURES_DIR = Path("/mnt/Scratch/phd_streamembed_features")

# STEP 2: Run all tiles through the model and save the resulting
# per-slide feature files (see utils/generate_features.py for the extraction
# + saving logic)
start = time()
features = generate_features(
    dataloader=dataloader,
    transform=transform,
    model=model,
    device=device,
    max_tiles_to_print=2,
    output_dir=FEATURES_DIR,
    model_encoder=MODEL_NAME,
    save_patch_features=False,
)
end = time()

# STEP 3: Report timing for this run
print(f"Time taken: {end - start:.3f} seconds")
print(f"Time per tile: {(end - start) / len(dataset):.3f} seconds")
print(f"Pyramid level: {PYRAMID_LEVEL}")


Generating features...
Printing first 2 tiles...


  0%|          | 0/45538 [00:05<?, ?it/s]


Tile #0
file: BRACS_1003658.svs
level=2 | x=19488 | y=448
CLS feature shape: (1536,)
CLS feature first 10: tensor([-0.0193,  0.1100, -0.0697, -0.5839,  0.1913, -0.1091, -0.0980,  0.7752,
         0.0624, -0.2652])
number of patches processed: 256
patch feature shape: (256, 1536)
patch #0 first 10 features: tensor([-0.1623,  0.2221, -0.1283, -0.4211,  0.2258,  0.0693,  0.1454,  0.2217,
         0.2513,  0.3187])
patch #1 first 10 features: tensor([-0.2184,  0.3242,  0.0348, -0.0558,  0.2392,  0.0862,  0.2020,  0.1616,
         0.3394,  0.5423])
patch #2 first 10 features: tensor([-0.3975,  0.2491,  0.1161,  0.1178,  0.1951,  0.1143,  0.1501,  0.1228,
         0.4935,  0.6823])
patch #3 first 10 features: tensor([-0.1034,  0.2664,  0.2732, -0.2405,  0.1514,  0.1451,  0.1136,  0.0197,
         0.2712,  0.3940])
patch #4 first 10 features: tensor([-0.0413, -0.0929, -0.0467, -0.0038,  0.3787,  0.1358,  0.1162,  0.2468,
        -0.3948,  0.4713])
patch #5 first 10 features: tensor([-0.1481,

## 4. Inspection of the features

### 4.1 One feature file
Summarizes one file: path, size, shapes, dtypes, finite check, value statistics and the stored tiling parameters. The figure shows the slide as a map coloured by the embeddings, the CLS norm per tile, and the first tile vectors and their similarity. Use it to confirm that a file is complete before running anything downstream.

In [ ]:
# STEP 1: Folder with the saved features (same as in the extraction cell, repeated so this section runs on its own)
FEATURES_DIR = Path("/mnt/Scratch/phd_streamembed_features")

# STEP 2: Summarize and plot one file. Pass feature_file="<path>.pt" to pick a specific slide;
# by default the first file of FEATURES_DIR / MODEL_NAME is used
_ = inspect_feature_file(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    dataset=dataset,   # only used for the tiling parameters of files that do not store them
    n_show=8,
)

### 4.2 CLS embeddings of all slides
Overview of the embedding space across slides: tile density and k-means clusters on the first two principal components, variance explained per component, cluster maps of three slides, cluster mix per slide, and tiles per slide.

The cosine-similarity panel compares random tile pairs from the same slide with pairs from different slides. If same-slide pairs are clearly more similar, slide-specific effects (staining, scanner) are present and should be considered when splitting data for classification.

In [ ]:
# STEP 1: Summarize the CLS embeddings of every slide in FEATURES_DIR / MODEL_NAME.
# max_slides=None uses all slides; n_clusters is limited to 3 (colour-blind-safe categorical colours)
overview = plot_cls_overview(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    max_slides=None,
    n_clusters=3,
)